In [ ]:
import os, json
import pandas as pd
import numpy as np

if os.path.exists('/workspace/data'):
    DATA_DIR, WORKSPACE_DIR = '/workspace/data', '/workspace'
elif os.path.exists('../environment/data'):
    DATA_DIR, WORKSPACE_DIR = '../environment/data', '..'
elif os.path.exists('environment/data'):
    DATA_DIR, WORKSPACE_DIR = 'environment/data', '.'
else:
    DATA_DIR, WORKSPACE_DIR = 'data', '.'

In [ ]:
picks_raw = pd.read_csv(f'{DATA_DIR}/picks.csv')
workers   = pd.read_csv(f'{DATA_DIR}/workers.csv')
shifts_df = pd.read_csv(f'{DATA_DIR}/shifts.csv')
sa        = pd.read_csv(f'{DATA_DIR}/shift_assignments.csv')
zones     = pd.read_csv(f'{DATA_DIR}/zones.csv')
za        = pd.read_csv(f'{DATA_DIR}/zone_assignments.csv')

total_raw_picks = int(len(picks_raw))
print(f'Raw picks: {total_raw_picks:,}')

In [ ]:
picks = picks_raw.drop_duplicates().reset_index(drop=True)
duplicate_picks_removed = int(total_raw_picks - len(picks))
print(f'Duplicates removed: {duplicate_picks_removed:,}')

In [ ]:
valid_wids = set(workers['worker_id']) | set(sa['worker_id'])
orphaned_mask = ~picks['worker_id'].isin(valid_wids)
orphaned_picks_removed = int(orphaned_mask.sum())
picks = picks[~orphaned_mask].reset_index(drop=True)
print(f'Orphaned picks removed: {orphaned_picks_removed:,}')

In [ ]:
sa['clock_in'] = pd.to_datetime(sa['clock_in_time'])
null_clockout_count = int(sa['clock_out_time'].isna().sum())

def sched_end(row):
    d = pd.to_datetime(row['date'])
    if row['shift_id'] == 'S1':
        return d.replace(hour=14, minute=0, second=0, microsecond=0)
    elif row['shift_id'] == 'S2':
        return d.replace(hour=22, minute=0, second=0, microsecond=0)
    else:  # S3 NIGHT ends next calendar day at 06:00
        return (d + pd.Timedelta(days=1)).replace(hour=6, minute=0, second=0, microsecond=0)

sa['sched_end'] = sa.apply(sched_end, axis=1)
sa['clock_out_eff'] = sa.apply(
    lambda r: pd.to_datetime(r['clock_out_time']) if pd.notna(r['clock_out_time']) else r['sched_end'],
    axis=1
)
print(f'NULL clock_out count: {null_clockout_count:,}')

In [ ]:
# NIGHT shift spans 22:00-06:00; picks after midnight carry the next day's date.
# Match each pick to the shift whose clock window contains pick_timestamp.
picks['pick_ts']        = pd.to_datetime(picks['pick_timestamp'])
picks['pick_date']      = picks['pick_ts'].dt.strftime('%Y-%m-%d')
picks['pick_date_prev'] = (picks['pick_ts'] - pd.Timedelta(days=1)).dt.strftime('%Y-%m-%d')

sa_cols = ['worker_id', 'date', 'assignment_id', 'shift_id',
           'clock_in', 'clock_out_eff', 'is_training_shift', 'break_duration_min']
p_cols  = ['pick_id', 'worker_id', 'pick_ts']

m_direct = (
    picks[p_cols + ['pick_date']]
    .merge(sa[sa_cols], left_on=['worker_id', 'pick_date'], right_on=['worker_id', 'date'], how='inner')
)
m_direct = m_direct[
    (m_direct['pick_ts'] >= m_direct['clock_in']) &
    (m_direct['pick_ts'] <= m_direct['clock_out_eff'])
]

sa_night = sa[sa['shift_id'] == 'S3'][sa_cols]
m_night  = (
    picks[p_cols + ['pick_date_prev']]
    .merge(sa_night, left_on=['worker_id', 'pick_date_prev'], right_on=['worker_id', 'date'], how='inner')
)
m_night = m_night[
    (m_night['pick_ts'] >= m_night['clock_in']) &
    (m_night['pick_ts'] <= m_night['clock_out_eff'])
]

out_cols = ['pick_id', 'assignment_id', 'shift_id', 'is_training_shift', 'break_duration_min']
matched  = (
    pd.concat([m_direct[out_cols], m_night[out_cols]], ignore_index=True)
    .drop_duplicates('pick_id')
)

out_of_window_picks_removed = int(len(picks) - len(matched))
picks = picks[picks['pick_id'].isin(matched['pick_id'])].merge(matched, on='pick_id', how='left')
print(f'Out-of-window picks removed: {out_of_window_picks_removed:,}')

In [ ]:
training_mask = picks['is_training_shift'].astype(bool)
training_picks_removed = int(training_mask.sum())
picks = picks[~training_mask].reset_index(drop=True)
print(f'Training picks removed: {training_picks_removed:,}')

In [ ]:
# picks.zone_id is the scanner's home zone at shift start.
# Reassignment windows in zone_assignments override zone attribution.
za['start_dt'] = pd.to_datetime(za['start_datetime'])
za['end_dt']   = pd.to_datetime(za['end_datetime'])

picks_za = picks[['pick_id', 'worker_id', 'pick_ts', 'zone_id']].merge(
    za[['worker_id', 'zone_id', 'start_dt', 'end_dt']].rename(columns={'zone_id': 'correct_zone'}),
    on='worker_id', how='left'
)
in_ra = (
    (picks_za['pick_ts'] >= picks_za['start_dt']) &
    (picks_za['pick_ts'] <= picks_za['end_dt'])
)
correct_zones = picks_za[in_ra][['pick_id', 'correct_zone']].drop_duplicates('pick_id')

picks = picks.merge(correct_zones, on='pick_id', how='left')
picks['zone_id_final'] = picks['correct_zone'].where(
    picks['correct_zone'].notna(), picks['zone_id']
)
picks = picks[picks['zone_id_final'].notna()].reset_index(drop=True)
print(f'Clean picks for analysis: {len(picks):,}')

In [ ]:
# Effective hours = clock duration - break_duration_min
sa['effective_hours'] = (
    (sa['clock_out_eff'] - sa['clock_in']).dt.total_seconds() / 3600
    - sa['break_duration_min'] / 60
).clip(lower=0)

# Split labor hours between home zone and reassigned zone(s) based on time in each.
sa_za = sa[['assignment_id', 'worker_id', 'shift_id',
            'clock_in', 'clock_out_eff', 'effective_hours']].merge(
    za[['worker_id', 'zone_id', 'start_dt', 'end_dt']],
    on='worker_id', how='left'
)
sa_za['za_s'] = sa_za[['start_dt', 'clock_in']].max(axis=1)
sa_za['za_e'] = sa_za[['end_dt',   'clock_out_eff']].min(axis=1)
sa_za['za_h'] = (
    (sa_za['za_e'] - sa_za['za_s']).dt.total_seconds() / 3600
).clip(lower=0)
sa_za_valid = sa_za[sa_za['za_h'] > 0].copy()

ra_per_sa = (
    sa_za_valid.groupby('assignment_id')['za_h'].sum()
    .reset_index().rename(columns={'za_h': 'total_ra_h'})
)
sa = sa.merge(ra_per_sa, on='assignment_id', how='left')
sa['total_ra_h'] = sa['total_ra_h'].fillna(0)
sa['home_h']     = (sa['effective_hours'] - sa['total_ra_h']).clip(lower=0)
sa = sa.merge(workers[['worker_id', 'home_zone_id']], on='worker_id', how='left')

home_labor = sa[['shift_id', 'home_zone_id', 'home_h']].copy()
home_labor.columns = ['shift_id', 'zone_id', 'labor_hours']
ra_labor = sa_za_valid[['shift_id', 'zone_id', 'za_h']].copy()
ra_labor.columns = ['shift_id', 'zone_id', 'labor_hours']

all_labor = pd.concat([home_labor, ra_labor], ignore_index=True)
all_labor = all_labor.merge(shifts_df[['shift_id', 'shift_name']], on='shift_id', how='left')
labor_agg = (
    all_labor.groupby(['zone_id', 'shift_name'])['labor_hours']
    .sum().reset_index()
    .rename(columns={'labor_hours': 'total_labor_hours'})
)

valid_labor_hours_total = round(float(labor_agg['total_labor_hours'].sum()), 2)
print(f'Total effective labor hours: {valid_labor_hours_total:,.2f}')

In [ ]:
picks = picks.merge(shifts_df[['shift_id', 'shift_name']], on='shift_id', how='left')

picks_agg = (
    picks.groupby(['zone_id_final', 'shift_name'])
    .size().reset_index(name='total_picks')
    .rename(columns={'zone_id_final': 'zone_id'})
)

report = picks_agg.merge(labor_agg, on=['zone_id', 'shift_name'], how='outer')
report['total_picks']        = report['total_picks'].fillna(0).astype(int)
report['total_labor_hours']  = report['total_labor_hours'].fillna(0).round(2)
report = report[(report['total_picks'] > 0) & (report['total_labor_hours'] > 0)].copy()
report['picks_per_labor_hour'] = (report['total_picks'] / report['total_labor_hours']).round(2)

report = report.merge(zones[['zone_id', 'zone_name']], on='zone_id', how='left')
report = report[['zone_id', 'zone_name', 'shift_name', 'total_picks',
                  'total_labor_hours', 'picks_per_labor_hour']]
report = report.sort_values('picks_per_labor_hour', ascending=False).reset_index(drop=True)

report.to_csv(f'{WORKSPACE_DIR}/productivity_report.csv', index=False)
print(f'Productivity report saved: {len(report)} zone-shift rows')
print(report.head(10).to_string(index=False))